# Notebook 12: On-device benchmark on hosted real Android phones (E15, H4 endpoint)

The registration planned `benchmark_model` on a physical phone over USB. This notebook measures the same frozen LiteRT files on **real phones hosted by Qualcomm AI Hub** instead: each file is uploaded and profiled on the device's CPU with the LiteRT (TFLite) runtime, and the service returns per-run inference times, load times and peak memory. No application is built. The change of device is written to `protocol_v3_deviations.json` (affects_results: False for accuracy; the latency figures come from hosted phones, named in the results).

**Files (10):** per crop, the Phase 1 deployed FP16 and full INT8 files, the v3 deployed FP16 (shift-augmented EfficientNet-B0), its full INT8 (k = 0), and the MobileNetV2 full-INT8 comparator.

**Before you run:** create a Qualcomm AI Hub account, copy your API token from the account settings, and add it in Colab → 🔑 Secrets as **`QAI_HUB_API_TOKEN`** with notebook access switched on. Do not paste the token into any cell.

**Runtime:** CPU is enough (the phones do the work). 20–60 minutes, mostly queue time on the service. Resumable: job IDs are saved, and rerunning collects finished jobs instead of resubmitting.

In [ ]:
# 1. Setup: Drive, token from Colab Secrets (never printed), client
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, time, glob, hashlib, datetime, subprocess
import numpy as np, pandas as pd
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'qai-hub'], check=True)
token = userdata.get('QAI_HUB_API_TOKEN')
assert token, 'Add QAI_HUB_API_TOKEN in Colab Secrets (key icon) and allow notebook access'
r = subprocess.run(['qai-hub', 'configure', '--api_token', token], capture_output=True, text=True); del token
assert r.returncode == 0, 'qai-hub configure failed: ' + r.stderr[-300:].replace('\n', ' ')
import qai_hub as hub
DRIVE = '/content/drive/MyDrive/ocdf'; OUT = f'{DRIVE}/records/nb12'; os.makedirs(OUT, exist_ok=True)
f3 = f'{DRIVE}/records/frozen_choices_v3.json'
assert hashlib.sha256(open(f3, 'rb').read()).hexdigest() == open(f'{DRIVE}/records/frozen_choices_v3.sha256').read().strip(), 'frozen_choices_v3 changed'
F3 = json.load(open(f3)); F2 = json.load(open(f'{DRIVE}/records/frozen_choices_v2.json'))
env = {'notebook': '12_device_benchmark_aihub', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version.split()[0],
       'qai_hub': str(getattr(hub, '__version__', 'unknown'))}
json.dump(env, open(f'{OUT}/environment_nb12.json', 'w'), indent=1); print(json.dumps(env, indent=1))

In [ ]:
# 2. The ten files (hash-checked against the frozen records) and the hosted Android devices
FILES = []
for crop in ['cassava', 'maize']:
    d1 = F2['deployed_instance'][crop]; d3 = F3['deployed_instance_v3'][crop]
    cand = [('phase1_fp16', d1['file'], d1['sha256']),
            ('phase1_int8', d1['file'].replace('_fp16.tflite', '_int8.tflite'), None),
            ('v3_fp16', d3['file'], d3['sha256']),
            ('v3_int8_k0', f'models_v3/{crop}/{crop}_{d3["model"]}_int8_k0.tflite', None),
            ('mnv2_int8', f'models_v3/{crop}/{crop}_shift_MobileNetV2_int8.tflite', None)]
    rec_sha = {r['file']: r['sha256'] for r in F3['all_files']} | {r['file']: r['sha256'] for r in F2['all_variants']}
    for role, rel, sha in cand:
        path = f'{DRIVE}/{rel}'; assert os.path.exists(path), f'missing {rel}'
        h = hashlib.sha256(open(path, 'rb').read()).hexdigest(); want = sha or rec_sha.get(rel)
        assert want, f'{rel} has no recorded hash'; assert h == want, f'{rel} changed after freezing'
        FILES.append({'crop': crop, 'role': role, 'file': rel, 'MiB': round(os.path.getsize(path) / 2**20, 3), 'sha256': h})
print(pd.DataFrame(FILES).to_string(index=False))
android = [d for d in hub.get_devices() if 'android' in str(getattr(d, 'os', '')).lower() or any('os:android' in a for a in getattr(d, 'attributes', []))]
names = sorted({d.name for d in android}); print(f'\n{len(names)} hosted Android devices available:'); print('\n'.join(names))
# Preference: a recent flagship and an older or mid-range phone (closer to what a farmer may own). Edit DEVICE_NAMES to choose others from the list above.
PREFER = ['Samsung Galaxy S24 (Family)', 'Samsung Galaxy S23 (Family)', 'Samsung Galaxy S21 (Family)', 'Samsung Galaxy A73 5G', 'Samsung Galaxy A53 5G', 'Google Pixel 3']
DEVICE_NAMES = [n for n in PREFER if n in names][:1] + [n for n in reversed(PREFER) if n in names][:1]
DEVICE_NAMES = list(dict.fromkeys(DEVICE_NAMES)) or names[:2]
print('\nusing:', DEVICE_NAMES)

In [ ]:
# 3. Submit profile jobs (CPU, LiteRT runtime), 1 and 4 threads; resumable via saved job IDs
jobs_path = f'{OUT}/jobs.json'; JOBS = json.load(open(jobs_path)) if os.path.exists(jobs_path) else {}
UPLOADED = {}
def options(threads):
    return f'--compute_unit cpu --tflite_options number_of_threads={threads}'
for dev in DEVICE_NAMES:
    for f in FILES:
        for threads in [1, 4]:
            key = f'{dev}|{f["crop"]}|{f["role"]}|t{threads}'
            if key in JOBS: continue
            if f['file'] not in UPLOADED: UPLOADED[f['file']] = hub.upload_model(f'{DRIVE}/{f["file"]}')
            try: job = hub.submit_profile_job(model=UPLOADED[f['file']], device=hub.Device(dev), options=options(threads), name=f'{f["crop"]}_{f["role"]}_t{threads}')
            except Exception as e:
                print('  thread option rejected (', str(e)[:80], '); submitting with the default thread count'); threads_used = 'default'
                job = hub.submit_profile_job(model=UPLOADED[f['file']], device=hub.Device(dev), options='--compute_unit cpu', name=f'{f["crop"]}_{f["role"]}_tdefault')
            else: threads_used = threads
            JOBS[key] = {'job_id': job.job_id, 'device': dev, **f, 'threads_requested': threads, 'threads_used': threads_used}
            json.dump(JOBS, open(jobs_path, 'w'), indent=1); print('submitted', key, job.job_id)
print(len(JOBS), 'jobs recorded')

In [ ]:
# 4. Collect results (waits for each job), parse latency and memory, deviation record, zip
def us_to_ms(x): return None if x is None else float(x) / 1000
rows = []
for key, j in JOBS.items():
    raw_path = f'{OUT}/profile_{j["job_id"]}.json'
    if os.path.exists(raw_path): prof = json.load(open(raw_path)); status = 'ok'
    else:
        job = hub.get_job(j['job_id']); st = job.wait()
        if not getattr(st, 'success', False):
            rows.append({**{k: j[k] for k in ['device', 'crop', 'role', 'file', 'MiB', 'threads_requested', 'threads_used']}, 'status': f'failed: {getattr(st, "message", st)}'[:200]}); continue
        prof = job.download_profile(); json.dump(prof, open(raw_path, 'w'), indent=1, default=str); status = 'ok'
    es = prof.get('execution_summary', {})
    times = es.get('all_inference_times') or []
    mem = es.get('inference_memory_peak_range') or [None, None]
    rows.append({**{k: j[k] for k in ['device', 'crop', 'role', 'file', 'MiB', 'threads_requested', 'threads_used']}, 'status': status, 'runs': len(times),
                 'median_ms': us_to_ms(np.median(times)) if times else us_to_ms(es.get('estimated_inference_time')),
                 'p90_ms': us_to_ms(np.percentile(times, 90)) if times else None, 'min_ms': us_to_ms(min(times)) if times else None, 'max_ms': us_to_ms(max(times)) if times else None,
                 'first_load_ms': us_to_ms(es.get('first_load_time')), 'warm_load_ms': us_to_ms(es.get('warm_load_time')),
                 'peak_memory_MB_low': None if mem[0] is None else mem[0] / 2**20, 'peak_memory_MB_high': None if mem[1] is None else mem[1] / 2**20,
                 'estimated_peak_memory_MB': None if es.get('estimated_inference_peak_memory') is None else es['estimated_inference_peak_memory'] / 2**20})
R = pd.DataFrame(rows); R.to_csv(f'{OUT}/device_benchmark.csv', index=False)
pd.set_option('display.width', 220)
show = [c for c in ['device', 'crop', 'role', 'MiB', 'threads_used', 'status', 'runs', 'median_ms', 'p90_ms', 'first_load_ms', 'peak_memory_MB_high'] if c in R]
print(R[show].round(2).to_string(index=False))
dev_info = {d.name: {'os': str(getattr(d, 'os', '')), 'attributes': list(getattr(d, 'attributes', []))} for d in hub.get_devices() if d.name in DEVICE_NAMES}
json.dump(dev_info, open(f'{OUT}/devices.json', 'w'), indent=1)
dev_path = f'{DRIVE}/records/protocol_v3_deviations.json'; devs = json.load(open(dev_path))
if not any(d.get('notebook') == '12' for d in devs):
    devs.append({'date_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'notebook': '12',
                 'what': "The registration planned the LiteRT benchmark_model binary over USB on a physical Android phone. The on-device measurement was made instead on "
                         f"real Android phones hosted by Qualcomm AI Hub ({', '.join(DEVICE_NAMES)}), CPU only, LiteRT runtime, 1 and 4 threads where the service accepted the option.",
                 'remedy': 'Device names and attributes are recorded in records/nb12/devices.json and every raw profile is kept. Latency and memory are reported as hosted-phone '
                           'measurements; the Mac offline benchmark  is unchanged. No accuracy figure depends on this change.',
                 'affects_results': False})
    json.dump(devs, open(dev_path, 'w'), indent=1); print('deviation recorded for Notebook 12')
import zipfile
zp = '/content/nb12_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{OUT}/*'):
        if os.path.isfile(p): z.write(p, os.path.basename(p))
    z.write(dev_path, os.path.basename(dev_path))
from google.colab import files
files.download(zp); print('nb12_results.zip ready')

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb12_results.zip`. The copies used for the published results are in this repository under `records/nb12/`.

If any job shows `failed`, rerun cell 4 once.